# 01 - TVM Core Concepts: Schedules, TIR, and Code Generation

In this notebook we explore the fundamental building blocks of TVM compilation:

- **IRModule / TIR**: the intermediate representation TVM operates on
- **Schedule**: a transformation layer that rewrites the loop structure *without* changing the mathematical result
- **Benchmarking**: measuring wall-clock performance with `time_evaluator`
- **Code generation**: lowering to LLVM IR or plain C

We use a 256×256 matrix multiplication as our running example — simple enough to reason about, yet large enough to show real performance differences.

Related material : [MLC Course Chapter 2](https://book.mlc.ai/chapter_tensor_program/index.html)

In [2]:
import numpy as np
import tvm
import tvm_ffi

from workloads import get_workload, list_workloads

print("Available workloads:", list_workloads())

Available workloads: ['conv2d_56', 'conv2d_bias_add_56', 'matmul_1024', 'matmul_256']


In [3]:
# Load the 256×256 matmul workload.
# get_workload() returns a tvm.IRModule — TVM's top-level container for one or
# more PrimFuncs written in Tensor IR (TIR).
mod: tvm.IRModule = get_workload("matmul_256")
print("Loaded:", type(mod))
mod.show()

Loaded: <class 'tvm.ir.module.IRModule'>


## What is TIR?

The output of `mod.show()` above is **Tensor IR (TIR)** — a low-level, loop-based representation.  
Key points:
- `T.grid(i, j, k)` defines three nested loops.
- `T.block("matmul")` names the computational block inside those loops.
- `T.axis.remap("SSR", ...)` labels axes as **S**patial (can be parallelized / reordered freely) or **R**eduction (order matters for correctness if we change the init).

The baseline loop order is **i → j → k**.  For matrix multiplication `C[i,j] += A[i,k] * B[k,j]`, walking `k` in the innermost loop means successive reads of `B[k, j]` jump across *columns* — poor cache behaviour on a row-major layout.

## What is a Schedule?

A `tvm.tir.Schedule` wraps an `IRModule` and exposes **primitive transformations** (reorder, split, tile, vectorize, …).  
These transforms rewrite the *loop structure* but never change the mathematical computation — the output is always identical.

```
Original:  for i  →  for j  →  for k   (ijk)
Reordered: for i  →  for k  →  for j   (ikj)  ← B[k,j] now accessed contiguously
```

In [27]:
# --- Build the BASELINE (ijk) schedule ---
# We create a Schedule from the unmodified IRModule.
# No transformations are applied, so this is our reference point.

sch_baseline: tvm.tir.Schedule = tvm.tir.Schedule(mod)

block_baseline: tvm.tir.schedule.schedule.BlockRV = sch_baseline.get_block("matmul")
i_b, j_b, k_b = sch_baseline.get_loops(block_baseline)

print("Baseline loop order:")
print(f"  outermost → {sch_baseline.get(i_b).loop_var.name}")
print(f"  middle    → {sch_baseline.get(j_b).loop_var.name}")
print(f"  innermost → {sch_baseline.get(k_b).loop_var.name}")

Baseline loop order:
  outermost → i
  middle    → j
  innermost → k


In [49]:
mod["main"].params

[a, b, c]

In [28]:
# --- Build the REORDERED (ikj) schedule ---
# We start from the *same* unmodified IRModule (schedules are not in-place on mod).
# Then we swap j and k, pushing the reduction axis (k) outward.

sch_reordered: tvm.tir.Schedule = tvm.tir.Schedule(mod)

block_reordered: tvm.tir.schedule.schedule.BlockRV = sch_reordered.get_block("matmul")
i_r, j_r, k_r = sch_reordered.get_loops(block_reordered)

# reorder() takes the desired order of loop variables
sch_reordered.reorder(i_r, k_r, j_r)  # ijk → ikj

print("Reordered loop order:")
i_r2, k_r2, j_r2 = sch_reordered.get_loops(block_reordered)
print(f"  outermost → {sch_reordered.get(i_r2).loop_var.name}")
print(f"  middle    → {sch_reordered.get(k_r2).loop_var.name}")
print(f"  innermost → {sch_reordered.get(j_r2).loop_var.name}")

Reordered loop order:
  outermost → i
  middle    → k
  innermost → j


## Inspecting the Modified TIR

`sch_reordered.mod` exposes the transformed `IRModule`.  
Compare it with `mod.show()` from the top — the only change is the order of the loop variables in `T.grid`.

In [29]:
print("=== BASELINE TIR (ijk) ===")
sch_baseline.mod.show()

print("\n=== REORDERED TIR (ikj) ===")
sch_reordered.mod.show()

=== BASELINE TIR (ijk) ===



=== REORDERED TIR (ikj) ===


## Compiling Both Versions

`tvm.build()` takes a scheduled `IRModule` and a `target` string, and returns an executable `Module`.  
We compile the baseline and the reordered variant independently so we can compare them.

In [30]:
TARGET: str = "llvm"

built_baseline:  tvm.runtime.Module = tvm.build(sch_baseline.mod,  target=TARGET)
built_reordered: tvm.runtime.Module = tvm.build(sch_reordered.mod, target=TARGET)

print("Baseline  runnable:", built_baseline.is_runnable())
print("Reordered runnable:", built_reordered.is_runnable())

Baseline  runnable: True
Reordered runnable: True


## Numerical Correctness

Before benchmarking, we verify that both kernels produce the same result as NumPy.  
This is the sanity-check habit you want to build **before** reading any performance numbers.

In [33]:
N: int = 256

np.random.seed(42)
A_np: np.ndarray = np.random.randn(N, N).astype("float32")
B_np: np.ndarray = np.random.randn(N, N).astype("float32")

# Reference result from NumPy
C_ref: np.ndarray = A_np @ B_np  # @ is the same as .dot in numpy syntax

# TVM tensors (zero-copy views via DLPack)
A_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(A_np)
B_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(B_np)

def run_matmul_kernel(built: tvm.runtime.Module) -> np.ndarray:
    """Allocate a fresh output buffer, run the kernel, return a numpy array."""
    C_np: np.ndarray = np.empty((N, N), dtype="float32")
    C_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(C_np)
    built(A_tvm, B_tvm, C_tvm)
    return C_tvm.numpy()

C_baseline:  np.ndarray = run_matmul_kernel(built_baseline)
C_reordered: np.ndarray = run_matmul_kernel(built_reordered)

rtol, atol = 1e-4, 1e-4
print("Baseline  matches NumPy:", np.allclose(C_baseline,  C_ref, rtol=rtol, atol=atol))
print("Reordered matches NumPy:", np.allclose(C_reordered, C_ref, rtol=rtol, atol=atol))

Baseline  matches NumPy: True
Reordered matches NumPy: True


## Benchmarking with `time_evaluator`

`time_evaluator` is TVM's built-in microbenchmark harness.  
It:
1. runs the kernel `repeat × number` times with proper warm-up,
2. returns a `BenchmarkResult` with mean / median / std in **seconds**.

> **`dev`** is the TVM device handle.  For CPU we use `tvm.cpu(0)`.

In [35]:
from tvm.runtime import profiling

dev: tvm.runtime.Device = tvm.cpu(0)

# Allocate output buffers on the TVM device for timing runs
A_np: np.ndarray = np.random.randn(N, N).astype("float32")
B_np: np.ndarray = np.random.randn(N, N).astype("float32")
C_np: np.ndarray = np.empty((N, N), dtype="float32")

C_buf_baseline : tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(C_np)
C_buf_reordered: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(C_np)
A_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(A_np)
B_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(B_np)

REPEAT: int  = 5   # independent timing runs
NUMBER: int  = 10  # kernel calls per run (averaged internally)

timer_baseline = built_baseline.time_evaluator(
    "main", dev, repeat=REPEAT, number=NUMBER
)
timer_reordered = built_reordered.time_evaluator(
    "main", dev, repeat=REPEAT, number=NUMBER
)

result_baseline  = timer_baseline(A_tvm,  B_tvm, C_buf_baseline)
result_reordered = timer_reordered(A_tvm, B_tvm, C_buf_reordered)

FLOPS: int = 2 * N**3  # multiply-adds

def report(name: str, result) -> None:
    mean_ms: float  = result.mean * 1e3
    std_ms: float   = result.std  * 1e3
    gflops: float   = FLOPS / result.mean / 1e9
    print(f"{name:<12}  mean={mean_ms:6.2f} ms  std={std_ms:5.2f} ms  {gflops:5.2f} GFLOPS")

report("baseline",  result_baseline)
report("reordered", result_reordered)
print(f"\nSpeedup: {result_baseline.mean / result_reordered.mean:.2f}x")

baseline      mean=  9.89 ms  std= 0.01 ms   3.39 GFLOPS
reordered     mean=  1.60 ms  std= 0.01 ms  20.96 GFLOPS

Speedup: 6.18x


## Compiling to C with `target="c"`

TVM is not tied to LLVM.  Switching the target string changes the entire code generation backend.  
`target="c"` emits standard C99 — useful for:

- understanding exactly what TVM generates at a readable level,
- deploying on microcontrollers or embedded targets with a plain C compiler,
- debugging or auditing the generated code.

`built_c.get_source()` returns the generated C as a plain string.

In [36]:
# Compile the BASELINE (ijk) to C
built_c_baseline: tvm.runtime.Module = tvm.build(sch_baseline.mod,  target="c")

# Compile the REORDERED (ikj) to C
built_c_reordered: tvm.runtime.Module = tvm.build(sch_reordered.mod, target="c")

In [44]:
c_baseline: str  = built_c_baseline.inspect_source()
c_reordered: str = built_c_reordered.inspect_source()

print("=== C SOURCE — BASELINE (ijk) ===\n")
print(c_baseline)

print("\n=== C SOURCE — REORDERED (ikj) ===\n")
print(c_reordered)

=== C SOURCE — BASELINE (ijk) ===

// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/base.h"
#include "tvm/runtime/c_backend_api.h"
#include "tvm/ffi/c_api.h"
#include <math.h>
#include <stdbool.h>
void* __tvm_ffi__library_ctx = NULL;
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t __tvm_ffi_main(void* self_handle, void* args, int32_t num_args, void* result);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t __tvm_ffi_main(void* self_handle, void* args, int32_t num_args, void* result) {
  int32_t a_type_index = (((TVMFFIAny*)args)[0].type_index);
  int32_t b_type_index = (((TVMFFIAny*)args)[1].type_index);
  int32_t c_type_index = (((TVMFFIAny*)args)[2].type_index);
  void* a = ((a_type_index == 70) ? ((void*)((char*)(((TVMFFIAny*)args)[0].v_ptr) + 24)) : (((TVMFFIAny*)args)[0].v_ptr));
  void* b = ((b_type_index == 70) ? ((void*)((char*)(((TVMFFIAny*)args)[1].v_ptr) + 24)) : (((TVMFFIAny*)args)[1].v_ptr));
  void* c = ((c_type_index == 70) ? ((void*)((char*

In [ ]:
# You can also export them in a c file
# with open("baseline_matmul.c", "w") as f:
#     f.write(c_baseline)
# with open("reordered_matmul.c", "w") as f:
#     f.write(c_reordered)

## Question: What can we say about the C-generated code ?

## Summary

| Step | API used | Key insight |
|---|---|---|
| Load workload | `get_workload()` → `tvm.IRModule` | TIR is the single source of truth |
| Create schedule | `tvm.tir.Schedule(mod)` | Wraps the module, no copy of data |
| Reorder loops | `sch.reorder(i, k, j)` | Pure structural transform, math unchanged |
| Inspect TIR | `sch.mod.show()` | Loop order is now visibly `ikj` |
| Compile | `tvm.build(sch.mod, target=...)` | One line from TIR to executable |
| Verify correctness | `np.allclose(C_tvm, C_ref)` | Always check before reading perf numbers |
| Benchmark | `built.time_evaluator(...)` | Proper warm-up, multiple repeats |
| C code generation | `target="c"` + `.inspect_source()` | Human-readable output for any C target |

**Next notebook:** loop tiling and vectorization — building on the same primitives to approach peak hardware throughput.